# Score Crediticio Interno

Este Score es heurístico porque aún no hay suficientes datos historicos para armar un modelo Probit


In [1]:
import pandas as pd
import numpy as np


### DATOS DEL SOLICITANTE

In [2]:
Sueldo = 13000
# Antigüedad laboral expresada en meses
Antiguedad_laboral = 12
# 0 = Es comisionista
# 1 = Tiene sueldo fijo
Ingreso_fijo = 0
Monto_solicitado = 7000
# Pago mensual que generaría el nuevo crédito
# Plazo solicitado en meses
Plazo_meses = 6

# Deudas/descuentos mensuales que ya tiene actualmente
Deuda_mensual_actual = 0

### Historial de credito interno
Opciones:
 "SIN_HISTORIAL"
 "BUENO"
 "ATRASOS_LEVES"
 "MALO"

In [3]:
Historial_crediticio = "SIN_HISTORIAL"

# Inflación anual vigente al momento de originar el crédito.
 EJEMPLO: 4% = 0.04

 Este dato posteriormente se puede actualizar automáticamente
 con la inflación vigente. 

In [4]:
Tasa_inflacion_anual = 0.04

# Margen fijo de la empresa:
Margen_interes = 0.02


# TASA ANUAL FIJA DEL CRÉDITO
Tasa_anual_credito = (
    Tasa_inflacion_anual
    + Margen_interes
)


# Convertimos la tasa anual efectiva a tasa mensual
Tasa_mensual_credito = (
    (1 + Tasa_anual_credito)**(1 / 12)
    - 1
)


print(
    "Tasa anual fija del crédito:",
    f"{Tasa_anual_credito:.2%}"
)

print(
    "Tasa mensual equivalente:",
    f"{Tasa_mensual_credito:.4%}"
)

Tasa anual fija del crédito: 6.00%
Tasa mensual equivalente: 0.4868%


Cálculo automático de la mensualidad

Ahora usamos la fórmula financiera de pago:

$$ PMT= P \left[ \frac{r(1+r)^n} {(1+r)^n-1} \right] $$

donde:

\(P\) = monto prestado
\(r\) = tasa mensual
\(n\) = número de pagos

In [5]:
def calcular_pago(monto, tasa_mensual, plazo_meses):

    if tasa_mensual == 0:
        return monto / plazo_meses

    pago = (
        monto
        *
        (
            tasa_mensual
            * (1 + tasa_mensual)**plazo_meses
        )
        /
        (
            (1 + tasa_mensual)**plazo_meses
            - 1
        )
    )

    return pago

In [6]:
Mensualidad_nuevo_credito = calcular_pago(
    Monto_solicitado,
    Tasa_mensual_credito,
    Plazo_meses
)


print(
    "Pago mensual:",
    f"{Mensualidad_nuevo_credito:,.2f}"
)

Pago mensual: 1,186.62


# Tabla de amortización

In [7]:
def tabla_amortizacion(
    monto,
    tasa_mensual,
    plazo_meses
):

    pago = calcular_pago(
        monto,
        tasa_mensual,
        plazo_meses
    )

    saldo = monto

    tabla = []

    for mes in range(1, plazo_meses + 1):

        # Interés del periodo
        interes = saldo * tasa_mensual

        # Abono a capital
        capital = pago - interes

        # Ajuste del último pago
        if mes == plazo_meses:
            capital = saldo
            pago_real = capital + interes
        else:
            pago_real = pago

        # Nuevo saldo
        saldo_final = saldo - capital

        # Evitamos errores de decimales
        saldo_final = max(saldo_final, 0)

        tabla.append({
            "Mes": mes,
            "Saldo_Inicial": saldo,
            "Pago": pago_real,
            "Interes": interes,
            "Capital": capital,
            "Saldo_Final": saldo_final
        })

        saldo = saldo_final

    return pd.DataFrame(tabla)

In [8]:
amortizacion = tabla_amortizacion(
    Monto_solicitado,
    Tasa_mensual_credito,
    Plazo_meses
)

amortizacion

,Mes,Saldo_Inicial,Pago,Interes,Capital,Saldo_Final
0,1,7000.000000,1186.622924,34.072854,1152.550070,5847.449930
1,2,5847.449930,1186.622924,28.462758,1158.160165,4689.289765
2,3,4689.289765,1186.622924,22.825355,1163.797569,3525.492197
3,4,3525.492197,1186.622924,17.160512,1169.462412,2356.029784
4,5,2356.029784,1186.622924,11.468094,1175.154829,1180.874955
5,6,1180.874955,1186.622924,5.747969,1180.874955,0.000000


### Variables calculadas o con formula

In [9]:
# ¿Ya cumplió 6 meses?
Cumplio_6_meses = 1 if Antiguedad_laboral >= 6 else 0

# Carga financiera después de otorgar el crédito
Carga_financiera = (
    Deuda_mensual_actual
    + Mensualidad_nuevo_credito
) / Sueldo

# Tamaño del crédito respecto al sueldo mensual
Credito_sueldo = Monto_solicitado / Sueldo

print("Cumplió 6 meses:", Cumplio_6_meses)
print("Carga financiera:", f"{Carga_financiera:.2%}")
print("Crédito / sueldo:", f"{Credito_sueldo:.2f} veces")

Cumplió 6 meses: 1
Carga financiera: 9.13%
Crédito / sueldo: 0.54 veces


Esto significa:

Antigüedad<6 meses⇒No elegible

In [10]:
if Cumplio_6_meses == 0:
    
    Elegible = False
    
else:
    
    Elegible = True

### Capacidad de pago
Por ejemplo:

$$ \frac{\text{Deudas actuales + mensualidad nueva}} {\text{Sueldo}} $$

Si la persona comprometería solamente 10% de su sueldo:

$$ 35/35 $$

Si comprometería más del 40%:

0/35

In [ ]:
if Carga_financiera <= 0.10:
    Puntos_capacidad = 35

elif Carga_financiera <= 0.20:
    Puntos_capacidad = 30

elif Carga_financiera <= 0.30:
    Puntos_capacidad = 20

elif Carga_financiera <= 0.40:
    Puntos_capacidad = 10

else:
    Puntos_capacidad = 0

### Antigüedad labotal
Max 15 puntos

In [12]:
if Antiguedad_laboral < 6:
    Puntos_antiguedad = 0

elif Antiguedad_laboral < 12:
    Puntos_antiguedad = 3

elif Antiguedad_laboral < 24:
    Puntos_antiguedad = 6

elif Antiguedad_laboral < 36:
    Puntos_antiguedad = 10

elif Antiguedad_laboral < 60:
    Puntos_antiguedad = 13

else:
    Puntos_antiguedad = 15

### Historial interno
Max 20 puntos
En este caso. lo puede poner Dani, Lupita o Rocio dependiento del criterio que tengas sobre la persona que va a pedir el prestamo en loq ue se obtiene más información de cómo paga.
Una persona sin historial no recibe cero, porque eso la castigaría por nunca haber solicitado antes.

Le damos un valor neutral:

10/20

In [13]:
if Historial_crediticio == "BUENO":
    Puntos_historial = 20

elif Historial_crediticio == "ATRASOS_LEVES":
    Puntos_historial = 12

elif Historial_crediticio == "SIN_HISTORIAL":
    Puntos_historial = 10

elif Historial_crediticio == "MALO":
    Puntos_historial = 0

else:
    Puntos_historial = 10

### Estabilidad de ingreso
Max 10 puntos donde
Ingreso fijo es lo siguiente:

0 = Comisionista --> si es comisionista se le da 3 úntos eso quiere decir que es más riesgoso
1 = Sueldo fijo


In [14]:
if Ingreso_fijo == 1:

    Puntos_estabilidad_ingreso = 10

else:

    Puntos_estabilidad_ingreso = 3

### Monto solcitado vs el sueldo
max 10 puntos

In [15]:
if Credito_sueldo <= 0.50:
    Puntos_monto = 10

elif Credito_sueldo <= 1:
    Puntos_monto = 8

elif Credito_sueldo <= 2:
    Puntos_monto = 5

elif Credito_sueldo <= 3:
    Puntos_monto = 2

else:
    Puntos_monto = 0

### Endeudamiento actual

In [ ]:
Endeudamiento_actual = Deuda_mensual_actual / Sueldo


if Endeudamiento_actual <= 0.10:
    Puntos_endeudamiento = 10

elif Endeudamiento_actual <= 0.20:
    Puntos_endeudamiento = 8

elif Endeudamiento_actual <= 0.30:
    Puntos_endeudamiento = 5

elif Endeudamiento_actual <= 0.40:
    Puntos_endeudamiento = 2

else:
    Puntos_endeudamiento = 0

### Score Total

In [17]:
Score = (
    Puntos_capacidad
    + Puntos_antiguedad
    + Puntos_historial
    + Puntos_estabilidad_ingreso
    + Puntos_endeudamiento
    + Puntos_monto
)

# Aseguramos que quede entre 0 y 100
Score = max(0, min(100, Score))


# ============================================================
# RESULTADO
# ============================================================

print("=" * 50)
print("SCORE CREDITICIO")
print("=" * 50)

print(f"Capacidad de pago:       {Puntos_capacidad}/35")
print(f"Antigüedad laboral:      {Puntos_antiguedad}/15")
print(f"Historial crediticio:    {Puntos_historial}/20")
print(f"Estabilidad de ingreso:  {Puntos_estabilidad_ingreso}/10")
print(f"Endeudamiento actual:    {Puntos_endeudamiento}/10")
print(f"Monto solicitado:        {Puntos_monto}/10")

print("-" * 50)

print(f"SCORE TOTAL:             {Score}/100")

print("=" * 50)

SCORE CREDITICIO
Capacidad de pago:       35/35
Antigüedad laboral:      6/15
Historial crediticio:    10/20
Estabilidad de ingreso:  3/10
Endeudamiento actual:    10/10
Monto solicitado:        8/10
--------------------------------------------------
SCORE TOTAL:             72/100


In [ ]:
if not Elegible:

    Decision = (
        "NO ELEGIBLE - "
        "ANTIGÜEDAD MENOR A 6 MESES"
    )

elif Score >= 80:

    Decision = "APROBAR"

elif Score >= 65:

    Decision = "APROBAR CONDICIONADO"

elif Score >= 50:

    Decision = "REVISION MANUAL"

elif Score >= 35:

    Decision = (
        "RIESGO ALTO - "
        "REVISAR MONTO O PLAZO"
    )

else:

    Decision = "RECHAZAR"

In [ ]:
Ingreso_fijo == 0
if (
    Ingreso_fijo == 0
    and Decision == "APROBAR"
):

    Decision = (
        "APROBAR CONDICIONADO - "
        "VALIDAR INGRESOS POR COMISIONES"
    )